### 📦 Cell 0 — Install the two libraries this notebook needs

`datasets` (Hugging Face's dataset-loading library) and `sentence-transformers` (the embedding model library) — everything else in this notebook builds on just these two.

In [1]:
!pip install datasets sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🧰 Cell 1 — Import the model, the trainer, and the loss function

`SentenceTransformer` loads the embedding model itself; `SentenceTransformerTrainer` and `SentenceTransformerTrainingArguments` handle the actual fine-tuning loop (batching, epochs, checkpoints); `CosineSimilarityLoss` is the training signal — it tells the model to pull two embeddings closer together or push them apart, based on cosine similarity, exactly the metric discussed earlier for semantic search.

In [2]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer,SentenceTransformerTrainingArguments
from sentence_transformers.losses import CosineSimilarityLoss

C:\Users\Avado\AppData\Local\Temp\ipykernel_44324\3157217311.py:3: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CosineSimilarityLoss


### 🔤 Cell 2 — Load a general-purpose embedding model, before any fine-tuning

`all-mpnet-base-v2` is a strong, general-purpose sentence embedding model — but "general-purpose" is exactly the problem from the *query-document distribution mismatch* failure mode covered earlier: it wasn't trained specifically on this domain's casual-question-vs-formal-answer phrasing, so it may not yet know they're related.

In [3]:
# 1. Load a model to finetune
model = SentenceTransformer("all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

find all models at https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

### 📏 Cell 4 — Baseline check: how close are these two sentences *before* fine-tuning?

Embeds a casual question ("What is the process of electrolysis?") and its correct-but-more-technical answer, then measures cosine similarity between them with the *un-tuned* model. This number is the baseline everything below is trying to improve — if it's not already close to 1, the model isn't yet good at recognizing that these two sentences are about the same thing.

In [4]:
# Calculate the similarity score between the two encoded sentences via the base model.

similarity = model.similarity(model.encode("What is the process of electrolysis?"),
                              model.encode("Electrolysis is a method of using a direct electric current to drive an otherwise non-spontaneous chemical reaction."))

print(similarity)

tensor([[0.7833]])


### 📚 Cell 5 — Load a real dataset of question-answer pairs to train on

`Mihaiii/qa-assistant` is a Hugging Face dataset of real query-and-answer pairs — this *is* the "real in-domain query-document pairs" mentioned as the requirement for contrastive fine-tuning earlier in this session's Q&A. Split into `train` (used to actually update the model's weights) and `test`/eval (used to check progress without letting the model see those examples during training).

In [5]:
# 2. Load a dataset to finetune on and split into train and val sets
dataset = load_dataset("Mihaiii/qa-assistant")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

### 👀 Cell 6 — Peek at the dataset's structure

Just displays `train_dataset` so you can see its shape and fields (how many examples, what columns exist) before training starts — a quick sanity check, not a training step.

In [6]:
train_dataset

Dataset({
    features: ['question', 'answer', 'score'],
    num_rows: 5771
})

### 🎯 Cell 7 — Attach the loss function to this specific model

`CosineSimilarityLoss(model)` is the actual training signal: for every training pair, it compares the model's *current* cosine similarity score against the *label* the dataset provides, and nudges the model's weights to close that gap. This is the literal mechanism behind "contrastive fine-tuning" — repeated many times across the whole dataset, it teaches the model that casually-phrased and formally-phrased versions of the same idea belong close together.

In [7]:
# 3. Define a loss function
loss = CosineSimilarityLoss(model)

More information about the loss at https://sbert.net/docs/package_reference/sentence_transformer/losses.html#cosinesimilarityloss

### ⚙️ Cell 9 — Configure how the training run itself behaves

This doesn't train anything yet — it just sets the dials: `num_train_epochs=5` (how many full passes over the training data), `per_device_train_batch_size=16` (how many examples get compared at once), `eval_steps=100` (check progress on the held-out set periodically, to catch overfitting), and `fp16=True` (use lower-precision math for speed — a standard trade-off since it barely affects final quality but noticeably speeds up training).

In [8]:
args = SentenceTransformerTrainingArguments(
    output_dir="models/finetuned-all-mpnet-base-v2", # Directory to save the fine-tuned model
    num_train_epochs=5, # Number of training epochs
    per_device_train_batch_size=16, # Batch size per device during training
    per_device_eval_batch_size=16, # Batch size per device during evaluation
    warmup_ratio=0.1, # Fraction of training steps used for a linear learning rate warmup
    fp16=True, # Use mixed precision training (FP16) for faster computations and reduced memory usage
    # Evaluation strategy: how often to evaluate the model
    eval_strategy="steps",  # Evaluate after a specified number of steps
    eval_steps=100, # Perform evaluation every 100 steps
    save_strategy="steps", # Save model checkpoint strategy: save after a specified number of steps
    save_steps=100, # Save model checkpoint every 100 steps
    save_total_limit=2, # Limit the total number of saved checkpoints to 2
    logging_steps=1, # Log training metrics every 1 step
    report_to="none" # Disable reporting to online trackers (e.g., WandB, TensorBoard)
)


The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


### 🏋️ Cell 10 — Actually run the fine-tuning loop

`trainer.train()` is where the real work happens: for 5 epochs, it repeatedly shows the model pairs from `train_dataset`, computes the cosine similarity loss from Cell 7, and updates the model's weights to reduce that loss — this is the expensive, time-consuming step the earlier "empirical tuning" and "labeled pairs are expensive" trade-offs were referring to.

In [ ]:
# 4. Create a trainer & train
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss
)

# One-time recovery: an earlier run hit a tooling timeout after saving checkpoint-600,
# so this run resumes training state (step, optimizer, scheduler) from that checkpoint
# instead of restarting from step 0.
trainer.train(resume_from_checkpoint="models/finetuned-all-mpnet-base-v2/checkpoint-600")


### 📈 Cell 11 — Same similarity check as Cell 4, now on the fine-tuned model

Runs the exact same two sentences from Cell 4 through the model again — but this time it's the *fine-tuned* version. Comparing this number against Cell 4's baseline is the direct, concrete evidence of whether fine-tuning actually worked: a higher similarity score means the model now recognizes the casual question and formal answer as closely related, which is exactly the query-document mismatch problem this whole notebook exists to fix.

In [ ]:
# Calculate the similarity score between the two encoded sentences via the finetuned model.

similarity = model.similarity(model.encode("What is the process of electrolysis?"),
                              model.encode("Electrolysis is a method of using a direct electric current to drive an otherwise non-spontaneous chemical reaction."))

print(similarity)

tensor([[0.9540]])


As seen here, the model is now able to calculate similarity between these two sentences more accurately.